# AI Professor — Indexação de Aula
Pipeline: `.mkv` → áudio → transcrição (Whisper) → chunks → embeddings → Qdrant

In [ ]:
# Instalar dependências
!pip install -q faster-whisper sentence-transformers qdrant-client
!apt-get install -y -q ffmpeg

In [ ]:
# Credenciais — configure como Secrets no Colab (cadeado na barra lateral)
from google.colab import userdata

QDRANT_URL = userdata.get('QDRANT_URL')
QDRANT_API_KEY = userdata.get('QDRANT_API_KEY')
COLLECTION = 'ai_professor_docs'
EMBED_MODEL = 'paraphrase-multilingual-mpnet-base-v2'
VECTOR_SIZE = 768

In [ ]:
# Upload do arquivo de vídeo
from google.colab import files
uploaded = files.upload()
video_path = list(uploaded.keys())[0]
print(f'Arquivo: {video_path}')

In [ ]:
# Extrair áudio mono 16kHz (formato ideal para Whisper)
import subprocess

audio_path = video_path.rsplit('.', 1)[0] + '.wav'
subprocess.run(
    ['ffmpeg', '-y', '-i', video_path,
     '-vn', '-acodec', 'pcm_s16le', '-ar', '16000', '-ac', '1',
     audio_path],
    check=True, capture_output=True
)
print(f'Áudio extraído: {audio_path}')

In [ ]:
# Transcrever com Whisper large-v3 (GPU)
from faster_whisper import WhisperModel

whisper = WhisperModel('large-v3', device='cuda', compute_type='float16')
segments_gen, info = whisper.transcribe(
    audio_path,
    language='pt',
    beam_size=5,
    vad_filter=True,       # remove silêncio
    word_timestamps=False,
)
segments = list(segments_gen)
print(f'Duração: {info.duration:.0f}s | Idioma detectado: {info.language} ({info.language_probability:.0%})')
print(f'Total de segmentos: {len(segments)}')

In [ ]:
# Agrupar segmentos em chunks de ~400 palavras com overlap de 50
def chunk_segments(segments, max_words=400, overlap_words=50):
    chunks = []
    current_words = []
    current_start = 0.0

    for seg in segments:
        words = seg.text.strip().split()
        if not current_words:
            current_start = seg.start
        current_words.extend(words)

        if len(current_words) >= max_words:
            chunks.append({
                'text': ' '.join(current_words),
                'start_sec': current_start,
                'end_sec': seg.end,
            })
            current_words = current_words[-overlap_words:]
            current_start = seg.start

    if current_words:
        last_end = segments[-1].end if segments else 0.0
        chunks.append({
            'text': ' '.join(current_words),
            'start_sec': current_start,
            'end_sec': last_end,
        })

    return chunks

chunks = chunk_segments(segments)
print(f'Total de chunks: {len(chunks)}')
print(f'Exemplo:\n{chunks[0]["text"][:300]}...')

In [ ]:
# Gerar embeddings
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer(EMBED_MODEL)
texts = [c['text'] for c in chunks]
embeddings = embed_model.encode(texts, batch_size=32, show_progress_bar=True, normalize_embeddings=True)
print(f'Shape dos embeddings: {embeddings.shape}')

In [ ]:
# Criar collection e indexar no Qdrant
import uuid
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams, PointStruct

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

# Recriar collection com dimensão correta
if client.collection_exists(COLLECTION):
    client.delete_collection(COLLECTION)
    print(f'Collection "{COLLECTION}" removida.')

client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE),
)
print(f'Collection "{COLLECTION}" criada (dim={VECTOR_SIZE}).')

# Indexar
points = [
    PointStruct(
        id=str(uuid.uuid4()),
        vector=embeddings[i].tolist(),
        payload={
            'text': chunks[i]['text'],
            'source': video_path,
            'start_sec': chunks[i]['start_sec'],
            'end_sec': chunks[i]['end_sec'],
        },
    )
    for i in range(len(chunks))
]

client.upsert(collection_name=COLLECTION, points=points)
print(f'\n✅ {len(points)} chunks indexados em "{COLLECTION}"')

In [ ]:
# Verificação — testar uma busca
query = 'Qual o tema principal da aula?'
q_vec = embed_model.encode([query], normalize_embeddings=True)[0].tolist()

results = client.search(collection_name=COLLECTION, query_vector=q_vec, limit=3)
for r in results:
    print(f'Score: {r.score:.3f} | {r.payload["text"][:150]}...')
    print()